# 02 — Data Cleaning

Notebook ini membersihkan dataset AQI Bangladesh mentah menjadi dataset yang siap untuk feature engineering.

## Input
- `../AQI Bangladesh.csv` — dataset mentah (1,048,551 baris × 13 kolom)

## Output
- `data/df_clean.csv` — data bersih (1,047,855 baris × 12 kolom)

## Langkah Utama
1. Parsing datetime & sorting per kota
2. Hapus duplikasi (subset `city_id` + `datetime`)
3. Konversi nilai negatif → NaN (batas fisik polutan tidak mungkin negatif)
4. Winsorizing outlier per kota (persentil 1%–99%)
5. Drop `carbon_dioxide` (>74% missing — tidak layak diimputasi)
6. Drop baris target AQI yang missing (target tidak boleh diimputasi)
7. Interpolasi linear per kota untuk polutan yang tersisa

**Prasyarat:** Dataset `AQI Bangladesh.csv` harus ada di root repo.
**Notebook berikutnya:** `03_feature_engineering.ipynb`


## 4. Data Cleaning

In [1]:
# === SEL INISIALISASI ===
import pandas as pd
import numpy as np
import os

file_local = '../AQI Bangladesh.csv'
if os.path.exists(file_local):
    print("Memuat dataset dari file lokal...")
    df = pd.read_csv(file_local)
else:
    print("Mengunduh dari Google Drive...")
    file_id = "1I7dtUfl3owj8kdwwwezEXtMBF2627_gk"
    df = pd.read_csv(f"https://drive.google.com/uc?id={file_id}")

pollutant_cols = ['pm10','pm2_5','carbon_monoxide','carbon_dioxide',
                  'nitrogen_dioxide','sulphur_dioxide','ozone','aqi']
print(f"Shape: {df.shape}")

Memuat dataset dari file lokal...
Shape: (1048551, 13)


### 4.1 Backup & Copy Working DataFrame

In [2]:
df_raw = df.drop(columns=['aqi_category'], errors='ignore').copy()  # simpan raw
df_clean = df.drop(columns=['aqi_category'], errors='ignore').copy()  # working copy

print(f'Shape awal: {df_clean.shape}')

Shape awal: (1048551, 13)


### 4.2 Parsing Datetime & Sorting

In [3]:
df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])

# Sort berdasarkan kota dan waktu
df_clean = df_clean.sort_values(['city_id', 'datetime']).reset_index(drop=True)

print(f'   Dtype datetime: {df_clean["datetime"].dtype}')
print(f'   Range: {df_clean["datetime"].min()} → {df_clean["datetime"].max()}')

   Dtype datetime: datetime64[us]
   Range: 2000-01-01 00:00:00 → 2025-11-23 23:00:00


### 4.3 Hapus Duplikasi

In [4]:
n_before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates(subset=['city_id', 'datetime'], keep='first')

n_after = df_clean.shape[0]
print(f'   Duplikasi dihapus sebanyak {n_before - n_after:,} baris')
print(f'   Shape setelah dihapus: {df_clean.shape}')

   Duplikasi dihapus sebanyak 0 baris
   Shape setelah dihapus: (1048551, 13)


### 4.4 Penanganan Nilai Negatif (Physical Bounds)

In [5]:
# Ganti nilai negatif dengan NaN

neg_report = {}
for col in pollutant_cols:
    mask = df_clean[col] < 0
    n_neg = mask.sum()
    if n_neg > 0:
        df_clean.loc[mask, col] = np.nan
        neg_report[col] = n_neg

if neg_report:
    print('Jumlah nilai negatif yang dikonversi ke NaN:')
    for col, n in neg_report.items():
        print(f'   {col}: {n:,} nilai')
else:
    print('Tidak ada nilai negatif yang ditemukan')

Jumlah nilai negatif yang dikonversi ke NaN:
   nitrogen_dioxide: 1 nilai
   ozone: 11 nilai


### 4.5 Penanganan Outlier (Winsorizing per Kota)

In [6]:
# Trimming extreme outlier per kota menggunakan percentile 1% dan 99%
# Pakai transform() agar kolom groupby tidak terdrop (pandas 3.x compatibility)

feature_cols = ['pm10', 'pm2_5', 'carbon_monoxide', 'carbon_dioxide',
                'nitrogen_dioxide', 'sulphur_dioxide', 'ozone']

for col in feature_cols:
    if col not in df_clean.columns:
        continue
    lo = df_clean.groupby('city_id')[col].transform(lambda x: x.quantile(0.01))
    hi = df_clean.groupby('city_id')[col].transform(lambda x: x.quantile(0.99))
    df_clean[col] = df_clean[col].clip(lower=lo, upper=hi)

print(f'Shape setelah winsorizing: {df_clean.shape}')

Shape setelah winsorizing: (1048551, 13)


### 4.6 Penanganan Missing Value

In [7]:
print('Status Missing Value Sebelum Imputasi')
mv = df_clean[pollutant_cols].isnull().sum()
mv_pct = (df_clean[pollutant_cols].isnull().mean() * 100).round(2)
print(pd.DataFrame({'Missing': mv, 'Missing%': mv_pct}).to_string())

Status Missing Value Sebelum Imputasi
                  Missing  Missing%
pm10                    0      0.00
pm2_5                   0      0.00
carbon_monoxide         0      0.00
carbon_dioxide     774327     73.85
nitrogen_dioxide        1      0.00
sulphur_dioxide         0      0.00
ozone                  11      0.00
aqi                   696      0.07


In [8]:
# Step 1: Drop carbon_dioxide karena terlalu banyak missing
CO2_MISSING_THRESHOLD = 50
cols_to_drop = [c for c in pollutant_cols
                if (df_clean[c].isnull().mean() * 100) > CO2_MISSING_THRESHOLD
                and c != 'aqi']

print(f'Kolom yang akan di-drop (>{CO2_MISSING_THRESHOLD}% missing): {cols_to_drop}')
df_clean = df_clean.drop(columns=cols_to_drop)
print(f'Shape setelah drop: {df_clean.shape}')

Kolom yang akan di-drop (>50% missing): ['carbon_dioxide']
Shape setelah drop: (1048551, 12)


In [9]:
# Step 2: Drop baris di mana AQI (target) missing
n_before = df_clean.shape[0]
df_clean = df_clean.dropna(subset=['aqi'])
print(f'Baris dengan AQI missing dihapus: {n_before - df_clean.shape[0]:,} baris')
print(f'Shape setelah drop: {df_clean.shape}')

Baris dengan AQI missing dihapus: 696 baris
Shape setelah drop: (1047855, 12)


In [10]:
# Step 3: Interpolasi linear per kota untuk fitur polutan
# Pakai transform() agar konsisten dengan src/cleaning.py dan kompatibel pandas 3.x

remaining_features = [c for c in ['pm10', 'pm2_5', 'carbon_monoxide',
                                    'nitrogen_dioxide', 'sulphur_dioxide', 'ozone']
                       if c in df_clean.columns]

for col in remaining_features:
    df_clean[col] = df_clean.groupby('city_id')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both').bfill().ffill()
    )

# Fallback dengan median global jika masih ada missing
for col in remaining_features:
    n_still_missing = df_clean[col].isnull().sum()
    if n_still_missing > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f'Fallback median untuk {col}: {n_still_missing} nilai')

In [11]:
print('Status Missing Value Setelah Imputasi')
print(df_clean[remaining_features + ['aqi']].isnull().sum().to_string())
print(f'\n Imputasi selesai. Shape: {df_clean.shape}')

Status Missing Value Setelah Imputasi
pm10                0
pm2_5               0
carbon_monoxide     0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
aqi                 0

 Imputasi selesai. Shape: (1047855, 12)


### 4.7 Validasi Akhir Data Cleaning

In [12]:
print('Summary Cleaning Reports')
print(f'Shape awal  : {df_raw.shape}')
print(f'Shape akhir : {df_clean.shape}')
print(f'Baris berkurang : {df_raw.shape[0] - df_clean.shape[0]:,}')
print(f'Kolom berkurang : {df_raw.shape[1] - df_clean.shape[1]}')
print(f'\nTotal missing value tersisa: {df_clean.isnull().sum().sum()}')

df_clean.info()

Summary Cleaning Reports
Shape awal  : (1048551, 13)
Shape akhir : (1047855, 12)
Baris berkurang : 696
Kolom berkurang : 1

Total missing value tersisa: 0
<class 'pandas.DataFrame'>
Index: 1047855 entries, 24 to 1048550
Data columns (total 12 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   city_id           1047855 non-null  int64         
 1   city_name         1047855 non-null  str           
 2   lat               1047855 non-null  float64       
 3   lon               1047855 non-null  float64       
 4   datetime          1047855 non-null  datetime64[us]
 5   pm10              1047855 non-null  float64       
 6   pm2_5             1047855 non-null  float64       
 7   carbon_monoxide   1047855 non-null  float64       
 8   nitrogen_dioxide  1047855 non-null  float64       
 9   sulphur_dioxide   1047855 non-null  float64       
 10  ozone             1047855 non-null  float64       
 11  aqi           

In [13]:
# === SIMPAN HASIL CLEANING ===
os.makedirs('data', exist_ok=True)
df_clean.to_csv('data/df_clean.csv', index=False)
print(f"df_clean disimpan -> data/df_clean.csv  {df_clean.shape}")

df_clean disimpan -> data/df_clean.csv  (1047855, 12)


---
## 5. Feature Engineering